# Rates — Simple CRUD Test

Insert, read, update, and delete rows from `[rates].[dim_curve]` and `[rates].[fact_observation]`.

In [ ]:
from datetime import datetime, timezone

import pandas as pd
from sqlalchemy import text

from imdr.config.settings import get_settings
from imdr.connectors.mssql import MSSQLConnector

connector = MSSQLConnector(get_settings())
print("Connected:", connector.engine.url)

## 1. READ — Check existing dim_curve rows

In [ ]:
with connector.session() as session:
    df_curves = pd.read_sql(
        text("SELECT id, ccy, curve, curve_type, curve_status FROM rates.dim_curve ORDER BY ccy, curve"),
        session.connection(),
    )

print(f"dim_curve rows: {len(df_curves)}")
df_curves.head(10)

## 2. CREATE — Insert a test observation

Pick the first active curve and insert a dummy par observation.

In [ ]:
# Use the first active curve_id from dim_curve
test_curve_id = int(df_curves.loc[df_curves["curve_status"] == "active", "id"].iloc[0])
test_ccy = df_curves.loc[df_curves["id"] == test_curve_id, "ccy"].iloc[0]
test_curve = df_curves.loc[df_curves["id"] == test_curve_id, "curve"].iloc[0]
print(f"Using curve_id={test_curve_id} ({test_ccy} {test_curve})")

insert_sql = text("""
    INSERT INTO rates.fact_observation (curve_id, ts, quote, tenor, value)
    OUTPUT INSERTED.id
    VALUES (:curve_id, :ts, :quote, :tenor, :value)
""")

params = dict(
    curve_id=test_curve_id,
    ts="2099-01-01T00:00:00+00:00",
    quote="par",
    tenor="5Y",
    value=3.85,
)

with connector.session() as session:
    result = session.execute(insert_sql, params)
    inserted_id = result.scalar()

print(f"Inserted row with id = {inserted_id}")

## 3. READ — Fetch the test row back

In [ ]:
read_sql = text("""
    SELECT o.id, c.ccy, c.curve, o.ts, o.quote, o.tenor, o.value, o.created_at
    FROM rates.fact_observation o
    JOIN rates.dim_curve c ON o.curve_id = c.id
    WHERE o.id = :id
""")

with connector.session() as session:
    df_row = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

df_row

## 4. UPDATE — Change the value

In [ ]:
update_sql = text("""
    UPDATE rates.fact_observation
    SET value = :value
    WHERE id = :id
""")

with connector.session() as session:
    session.execute(update_sql, {"value": 4.25, "id": inserted_id})

# Verify
with connector.session() as session:
    df_updated = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Updated value = {df_updated['value'].iloc[0]}")
df_updated

## 5. DELETE — Remove the test row

In [ ]:
delete_sql = text("DELETE FROM rates.fact_observation WHERE id = :id")

with connector.session() as session:
    result = session.execute(delete_sql, {"id": inserted_id})

print(f"Deleted {result.rowcount} row(s)")

# Verify
with connector.session() as session:
    df_check = pd.read_sql(read_sql, session.connection(), params={"id": inserted_id})

print(f"Rows remaining for id={inserted_id}: {len(df_check)}")

In [ ]:
connector.dispose()
print("Done — connection pool closed.")